# Ejercicio 4. Flujos no autorizados y bloqueo de tráfico

Curso CIB-209, Temas Especiales en Seguridad de Datos y Sistemas.

Modificar únicamente la celda CONFIGURACIÓN. Ejecutar después todas las celdas en orden.

## Recordatorio de la técnica

Este ejercicio combina dos técnicas distintas, en dos etapas.

La etapa 1 no aprende nada. Compara cada conexión observada contra la matriz de flujos aprobada, exigiendo coincidencia exacta en cuatro campos: segmento de origen, segmento de destino, puerto y protocolo. Lo que no está en la matriz queda separado como no autorizado. Ese es el modelo de denegación por omisión.

La etapa 2 sí usa aprendizaje automático supervisado. Una máquina de vectores de soporte se entrena con 1200 conexiones ya etiquetadas como benignas o maliciosas y busca la frontera que mejor separa las dos clases. Para una conexión nueva, la confianza que reporta indica de qué lado de esa frontera cae y qué tan lejos está de ella, expresado entre 0 y 1. No es una probabilidad de culpabilidad ni una prueba: es la distancia a una frontera aprendida de ejemplos pasados.

## Vocabulario

- White list: lista de lo que está expresamente permitido.
- Tupla: combinación de origen, destino, puerto y protocolo que se compara contra la white list.
- Denegación por omisión: lo que no está permitido queda prohibido.
- Frontera de decisión: separación que el modelo aprendió entre las dos clases.
- Confianza: qué tan lejos de la frontera cae la conexión, del lado malicioso.
- Umbral: valor de confianza a partir del cual se bloquea. Es una decisión de negocio.

Este recordatorio explica la técnica y cómo leer las salidas. No interpreta los resultados, eso le corresponde al grupo.

In [ ]:
# ======================= CONFIGURACION =======================
NUMERO_DE_GRUPO = "G00"

matriz_de_flujos_autorizados = "original"   # opciones: "original", "corregida"
umbral_de_confianza = 0.5                   # corridas solicitadas: 0.5, 0.5, 0.8
# =============================================================

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def sello_de_corrida(**parametros):
    texto = "|".join(f"{k}={parametros[k]}" for k in sorted(parametros))
    return hashlib.md5(texto.encode()).hexdigest()[:4].upper()

ATRIBUTOS = ["volumen_mb_sesion", "frecuencia_dia", "hora_predominante",
             "duracion_seg", "destino_externo", "puerto_no_estandar"]

archivo_matriz = ("flujos_autorizados.csv" if matriz_de_flujos_autorizados == "original"
                  else "flujos_autorizados_corregida.csv")
autorizados = pd.read_csv(archivo_matriz)
observadas = pd.read_csv("conexiones_observadas.csv")
historial = pd.read_csv("historial_conexiones_etiquetado.csv")

print("Grupo:", NUMERO_DE_GRUPO)
print("matriz_de_flujos_autorizados =", matriz_de_flujos_autorizados, "  archivo:", archivo_matriz)
print("umbral_de_confianza =", umbral_de_confianza)
print("Sello de la corrida:", sello_de_corrida(matriz=matriz_de_flujos_autorizados, umbral=umbral_de_confianza))
print()
print("Conexiones observadas:", len(observadas))
print("Flujos en la matriz aprobada:", len(autorizados))
print("Registros del historial etiquetado:", len(historial))
print()
print("MATRIZ DE FLUJOS AUTORIZADOS")
print(autorizados.to_string(index=False))

In [ ]:
llaves = set(zip(autorizados.segmento_origen, autorizados.segmento_destino,
                 autorizados.puerto, autorizados.protocolo))
coincide = [t in llaves for t in zip(observadas.segmento_origen, observadas.segmento_destino,
                                     observadas.puerto, observadas.protocolo)]
observadas["en_la_matriz"] = coincide
no_autorizadas = observadas[~observadas.en_la_matriz].copy()

print("ETAPA 1. FILTRADO POR COINCIDENCIA EXACTA CONTRA LA WHITE LIST")
print("Conexiones que coinciden con la matriz aprobada:", int(observadas.en_la_matriz.sum()))
print("Conexiones sin coincidencia, pasan a la etapa 2:", len(no_autorizadas))
print()
print("CONEXIONES NO AUTORIZADAS AGRUPADAS POR SERVICIO DE NEGOCIO")
print(no_autorizadas.servicio_negocio.value_counts().to_string())

In [ ]:
modelo = make_pipeline(StandardScaler(), SVC(kernel="linear", probability=True, random_state=0))
modelo.fit(historial[ATRIBUTOS], (historial.etiqueta == "maliciosa").astype(int))

no_autorizadas["confianza_maliciosa"] = modelo.predict_proba(no_autorizadas[ATRIBUTOS])[:, 1].round(3)
no_autorizadas["decision"] = np.where(
    no_autorizadas.confianza_maliciosa >= umbral_de_confianza, "bloquear", "permitir")

print("ETAPA 2. CLASIFICACIÓN DE LAS CONEXIONES NO AUTORIZADAS")
columnas = ["id_conexion", "segmento_origen", "segmento_destino", "puerto", "protocolo",
            "volumen_mb_sesion", "frecuencia_dia", "hora_predominante",
            "servicio_negocio", "confianza_maliciosa", "decision"]
print(no_autorizadas.sort_values("confianza_maliciosa", ascending=False)[columnas].to_string(index=False))

In [ ]:
reglas = no_autorizadas.sort_values("confianza_maliciosa", ascending=False).assign(
    accion=lambda d: np.where(d.decision == "bloquear", "DENY", "ALLOW"))
reglas_tabla = reglas[["accion", "segmento_origen", "segmento_destino", "puerto",
                       "protocolo", "servicio_negocio", "confianza_maliciosa"]]
print("REGLAS GENERADAS PARA EL CORTAFUEGOS")
print(reglas_tabla.to_string(index=False))
print()
print("Reglas generadas en total:", len(reglas_tabla))
print("Reglas DENY:", int((reglas_tabla.accion == 'DENY').sum()),
      " Reglas ALLOW:", int((reglas_tabla.accion == 'ALLOW').sum()))

In [ ]:
bloqueadas = no_autorizadas[no_autorizadas.decision == "bloquear"]
permitidas = no_autorizadas[no_autorizadas.decision == "permitir"]
servicios_interrumpidos = sorted(set(bloqueadas[bloqueadas.etiqueta_real == "benigna"].servicio_negocio))
maliciosas_que_pasaron = permitidas[permitidas.etiqueta_real == "maliciosa"]

print("RESUMEN DE IMPACTO")
print("Grupo:", NUMERO_DE_GRUPO, " Sello:", sello_de_corrida(matriz=matriz_de_flujos_autorizados, umbral=umbral_de_confianza))
resumen = pd.DataFrame([
    ["Conexiones no autorizadas detectadas en la etapa 1", len(no_autorizadas)],
    ["Conexiones bloqueadas", len(bloqueadas)],
    ["Conexiones benignas bloqueadas", int((bloqueadas.etiqueta_real == "benigna").sum())],
    ["Conexiones maliciosas que quedaron permitidas", len(maliciosas_que_pasaron)],
    ["Reglas generadas", len(no_autorizadas)],
], columns=["indicador", "valor"])
print(resumen.to_string(index=False))
print()
print("SERVICIOS DE NEGOCIO INTERRUMPIDOS POR EL BLOQUEO")
if servicios_interrumpidos:
    for s in servicios_interrumpidos:
        print(" -", s)
else:
    print(" ninguno")
print()
print("CONEXIONES MALICIOSAS QUE QUEDARON PERMITIDAS")
if len(maliciosas_que_pasaron):
    print(maliciosas_que_pasaron[["id_conexion", "segmento_origen", "segmento_destino", "puerto",
                                  "volumen_mb_sesion", "frecuencia_dia", "hora_predominante",
                                  "confianza_maliciosa"]].to_string(index=False))
else:
    print(" ninguna")

## Cómo se lee el gráfico de barras

Cada barra horizontal es una conexión no autorizada, identificada a la izquierda por su código y por el servicio de negocio al que pertenece. El largo de la barra es la confianza de que la conexión sea maliciosa, de 0 a 1.

La línea negra vertical es el umbral configurado. Las barras rojas son las que quedaron a la derecha del umbral y se bloquean. Las grises quedaron por debajo y se permiten.

Lo que conviene revisar es el nombre del servicio de cada barra roja, para ver si lo que se corta es tráfico sin dueño o un servicio del que depende la operación, y también las barras grises más largas, que son las conexiones que el control dejó pasar aunque estuvieran cerca del umbral.

In [ ]:
orden = no_autorizadas.sort_values("confianza_maliciosa")
etiquetas = [f"{a} ({b[:28]})" for a, b in zip(orden.id_conexion, orden.servicio_negocio)]
colores = ["#b91c1c" if d == "bloquear" else "#94a3b8" for d in orden.decision]
fig, ax = plt.subplots(figsize=(9.5, 0.32 * len(orden) + 1.5))
ax.barh(range(len(orden)), orden.confianza_maliciosa, color=colores)
ax.set_yticks(range(len(orden)))
ax.set_yticklabels(etiquetas, fontsize=7)
ax.axvline(umbral_de_confianza, color="#111827", linestyle="--",
           label=f"umbral_de_confianza = {umbral_de_confianza}")
ax.set_xlabel("confianza de que la conexión es maliciosa")
ax.set_xlim(0, 1)
ax.set_title(f"Grupo {NUMERO_DE_GRUPO} | matriz_de_flujos_autorizados={matriz_de_flujos_autorizados} | umbral_de_confianza={umbral_de_confianza} | sello={sello_de_corrida(matriz=matriz_de_flujos_autorizados, umbral=umbral_de_confianza)}")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()